In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import transformers
from datasets import load_from_disk
from tqdm.auto import tqdm

from experiments.jlens_readout_sanity.constants import MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa import deduplicate, normalize_rows
from jlens_reasoning.evaluation import (
    GenerationStatus,
    ModelOutput,
    evaluate_paper_binary,
)

OUTPUT_DIR = context.runs_dir / "flenqa-accuracy"
RESULT_PATH = OUTPUT_DIR / "results.parquet"
LENGTHS = (250, 500, 1000, 2000, 3000)
EXPECTED_UNIQUE_COUNTS = {
    250: 300,
    500: 2_368,
    1000: 2_394,
    2000: 2_400,
    3000: 2_400,
}
MAX_SEQ_LEN = 4096
MAX_NEW_TOKENS = 64

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["train"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
prompts = deduplicate(rows)
assert len(rows) == 12_000
assert len(prompts) == 9_862
{"source_rows": len(rows), "unique_prompts": len(prompts)}

In [ ]:
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()

In [ ]:
def generate_output(prompt: str) -> tuple[int, ModelOutput]:
    encoded = tokenizer(prompt, return_tensors="pt", truncation=False)
    input_ids = encoded["input_ids"].to(context.device)
    n_input_tokens = int(input_ids.shape[1])
    if n_input_tokens > MAX_SEQ_LEN:
        raise ValueError(f"Prompt exceeds {MAX_SEQ_LEN} tokens: {n_input_tokens}")
    attention_mask = encoded["attention_mask"].to(context.device)
    with torch.inference_mode():
        generated = causal_lm.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    generated_ids = generated[0, n_input_tokens:].tolist()
    eos_ids = causal_lm.generation_config.eos_token_id
    eos_token_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or ())
    complete = bool(generated_ids and generated_ids[-1] in eos_token_ids)
    scored_ids = generated_ids[:-1] if complete else generated_ids
    output = ModelOutput(
        text=tokenizer.decode(scored_ids, skip_special_tokens=True),
        token_ids=tuple(generated_ids),
        token_pieces=tuple(
            tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
            for token_id in generated_ids
        ),
        generation_status=(
            GenerationStatus.COMPLETE if complete else GenerationStatus.TRUNCATED
        ),
        finish_reason="eos" if complete else "length",
    )
    return n_input_tokens, output

In [ ]:
records = []
for prompt in tqdm(prompts, desc="FLenQA accuracy prompts", unit="prompt"):
    ctx_sizes = {item.ctx_size for item in prompt.provenance}
    if len(ctx_sizes) != 1:
        raise ValueError(f"Prompt spans nominal lengths: {sorted(ctx_sizes)}")
    n_input_tokens, output = generate_output(prompt.text)
    evaluation = evaluate_paper_binary(output, expected=prompt.label)
    records.append(
        {
            "prompt_id": prompt.prompt_id,
            "problem_id": prompt.problem_id,
            "task": prompt.task,
            "label": prompt.label,
            "text": prompt.text,
            "ctx_size": ctx_sizes.pop(),
            "n_input_tokens": n_input_tokens,
            "paper_weight": sum(
                item.dispersion == "random" for item in prompt.provenance
            ),
            "model_name": MODEL_NAME,
            "code_revision": PROJECT_COMMIT,
            "generated_token_ids": list(output.token_ids),
            "generated_token_pieces": list(output.token_pieces),
            "generated_text": output.text,
            "generation_status": output.generation_status.value,
            "finish_reason": output.finish_reason,
            "verdict": evaluation.verdict,
            "correct": evaluation.correct,
        }
    )

assert len(records) == 9_862
actual_counts = Counter(record["ctx_size"] for record in records)
assert dict(actual_counts) == EXPECTED_UNIQUE_COUNTS
paper_counts = Counter()
for record in records:
    paper_counts[record["ctx_size"]] += record["paper_weight"]
assert dict(paper_counts) == {length: 600 for length in LENGTHS}
{"rows": len(records), "counts_by_length": dict(actual_counts)}

In [ ]:
RESULT_SCHEMA = pa.schema(
    [
        pa.field("prompt_id", pa.string(), nullable=False),
        pa.field("problem_id", pa.int32(), nullable=False),
        pa.field("task", pa.string(), nullable=False),
        pa.field("label", pa.bool_(), nullable=False),
        pa.field("text", pa.string(), nullable=False),
        pa.field("ctx_size", pa.int32(), nullable=False),
        pa.field("n_input_tokens", pa.int32(), nullable=False),
        pa.field("paper_weight", pa.int16(), nullable=False),
        pa.field("model_name", pa.string(), nullable=False),
        pa.field("code_revision", pa.string(), nullable=False),
        pa.field("generated_token_ids", pa.list_(pa.int32()), nullable=False),
        pa.field("generated_token_pieces", pa.list_(pa.string()), nullable=False),
        pa.field("generated_text", pa.string(), nullable=False),
        pa.field("generation_status", pa.string(), nullable=False),
        pa.field("finish_reason", pa.string()),
        pa.field("verdict", pa.bool_()),
        pa.field("correct", pa.bool_(), nullable=False),
    ]
)
results = pa.Table.from_pylist(records, schema=RESULT_SCHEMA)
assert results.num_rows == 9_862
assert RESULT_PATH.name == "results.parquet"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pq.write_table(results, RESULT_PATH, compression="zstd")
frame = results.to_pandas()
RESULT_PATH

In [ ]:
frame["weighted_correct"] = frame["correct"] * frame["paper_weight"]
frame["weighted_missing"] = frame["verdict"].isna() * frame["paper_weight"]
paper_summary = (
    frame.groupby("ctx_size", as_index=False)
    .agg(
        correct=("weighted_correct", "sum"),
        total=("paper_weight", "sum"),
        no_verdict=("weighted_missing", "sum"),
    )
    .sort_values("ctx_size")
)
paper_summary["accuracy"] = paper_summary["correct"] / paper_summary["total"]
assert paper_summary["ctx_size"].tolist() == list(LENGTHS)
assert paper_summary["total"].tolist() == [600] * len(LENGTHS)
display(paper_summary)
plt.figure(figsize=(8, 4.5))
plt.plot(
    paper_summary["ctx_size"],
    paper_summary["accuracy"],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — paper weighting")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
unique_summary = (
    frame.groupby("ctx_size", as_index=False)
    .agg(
        correct=("correct", "sum"),
        total=("prompt_id", "size"),
        no_verdict=("verdict", lambda values: values.isna().sum()),
    )
    .sort_values("ctx_size")
)
unique_summary["accuracy"] = unique_summary["correct"] / unique_summary["total"]
assert (
    dict(zip(unique_summary["ctx_size"], unique_summary["total"], strict=True))
    == EXPECTED_UNIQUE_COUNTS
)
display(unique_summary)
plt.figure(figsize=(8, 4.5))
plt.plot(
    unique_summary["ctx_size"],
    unique_summary["accuracy"],
    marker="o",
)
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA accuracy by input length — unique prompts")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
task_summary = (
    frame.groupby(["task", "ctx_size"], as_index=False)
    .agg(correct=("correct", "sum"), total=("prompt_id", "size"))
    .sort_values(["task", "ctx_size"])
)
task_summary["accuracy"] = task_summary["correct"] / task_summary["total"]
display(task_summary)
plt.figure(figsize=(8, 4.5))
for task, task_frame in task_summary.groupby("task", sort=True):
    plt.plot(
        task_frame["ctx_size"],
        task_frame["accuracy"],
        marker="o",
        label=task,
    )
plt.xticks(LENGTHS)
plt.ylim(0, 1)
plt.xlabel("Input length (# nominal tokens)")
plt.ylabel("Accuracy")
plt.title("FLenQA unique-prompt accuracy by task")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
verdict_labels = (
    frame["verdict"].map({True: "True", False: "False"}).fillna("No verdict")
)
verdict_counts = pd.crosstab(frame["ctx_size"], verdict_labels).reindex(
    index=LENGTHS, columns=["True", "False", "No verdict"], fill_value=0
)
token_lengths = (
    frame.groupby("ctx_size")["n_input_tokens"]
    .agg(["min", "median", "max"])
    .reindex(LENGTHS)
)
display("Verdict counts by nominal length", verdict_counts)
display("Exact Qwen token lengths by nominal bucket", token_lengths)